#### LSTM 자연어 모델 문제
1. data 폴더 안에 ratings_train.txt 파일을 로드
2. 텍스트 정규화 함수를 이용하여 document 컬럼의 텍스트들을 정규화
    - 특수문자 제거, 2칸 이상의 공백을 1칸의 공백으로 대체, 문자 좌우의 공백을 제거
3. 공백 텍스트 제거
4. 결측치 제거
5. 중복된 데이터 제거
6. 상위 5000개의 데이터를 추출
7. komoran을 이용해서 데이터 토큰화
    - 품사 : NNP, NNG, VV, VA, MAG, SL만 사용
8. 단어 사전을 생성한다. (최소 출현 횟수는 2회)
8. Dataset을 생성(인코딩 작업 결합)하고 collate_fn 생성하여 패딩 토큰을 채워준다.
9. DataLoader를 생성
10. 8 : 2 의 비율로 train, vali 데이터셋으로 나눠준다.
10. LSTM 학습 모델 생성
    - Embedding() -> LSTM() -> Linear()
    - LSTM에서는 마지막 히든층을 이용하여 선형 모델에 대입
11. 에폭의 횟수는 50회로 모델을 검증

In [1]:
import pandas as pd 
import torch 
import torch.nn as nn
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from konlpy.tag import Komoran
from collections import Counter
from tqdm import tqdm
import re 

In [2]:
#1. 
df = pd.read_csv("./ratings_train.txt", sep='\t')

In [3]:
#2. 텍스트 정규화 함수
def normalize(text):
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text
    
df['document'] = df['document'].map(normalize)

In [4]:
#3.
df = df.loc[ ~(df['document'] == ''),  ]
df.info()

<class 'pandas.DataFrame'>
Index: 149564 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        149564 non-null  int64
 1   document  149564 non-null  str  
 2   label     149564 non-null  int64
dtypes: int64(2), str(1)
memory usage: 4.6 MB


In [5]:
#4. 
df.dropna(inplace=True)
#5. 
df.drop_duplicates('document',inplace=True)

In [6]:
#7. 
komoran = Komoran()

def tokenize(text):
    allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'SL']

    result = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            result.append(word)
    return result

tokenized_setences = [ tokenize(text) for text in df['document'] ]

In [7]:
#6.
df=df[:5000]
len(df)

5000

In [ ]:
#8.
#단어 사전 구축 
vocab = {
    '<PAD>' : 0, 
    '<UNK>' : 1
}
all_tokens = [ token for tokens in tokenized_setences for token in tokens ]
token_count = Counter(all_tokens)

for token, count in token_count.items():
    if count >= 2:
        vocab[token] = len(vocab)
vocab

In [9]:
#9.
#Dataset, collate_fn, DataLoader 셍성
#Dataset을 선언
class LSTMDataset(Dataset):
    def __init__(self, tokenized_texts, labels, vocab):
        self.labels = labels.values
        self.data = [
            [vocab.get(token, vocab['<UNK>']) for token in tokens] 
            for tokens in tokenized_texts
        ]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

#Dataset 셍성
dataset = LSTMDataset(tokenized_setences, df['label'], vocab)

#11.
# train의 길이와 test의 길이를 설정
train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [10]:
#9.
def collate_fn(batch):
    # text_list = [item[0] for item in batch]
    # labels = [item[1] for item in batch]
    text_list, labels = zip(*batch)
    padded_texts = pad_sequence(text_list, batch_first=True, padding_value=vocab['<PAD>'])
    labels = torch.tensor(labels, dtype=torch.long)
    return padded_texts, labels

In [11]:
#11.
train_loader = DataLoader(train_dataset, batch_size=64, shuffle = True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle = True, collate_fn= collate_fn)

In [12]:
#12.
class LSTMCLF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_classes, dropout = 0.5, head_type = 'last'):
        super().__init__()

        self.head_type = head_type

        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=vocab['<PAD>'])

        # 자비에르 초기화 
        torch.nn.init.xavier_uniform_(self.emb.weight)

        self.lstm = nn.LSTM(emb_dim, hidden_size, batch_first=True)
        # 과적합 방지용 dropout
        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        embedding = self.emb(x)

        # LSTM 결과 값 A, (B,C)
        lstm_out, (hidden, cell) = self.lstm(embedding)
        if self.head_type == 'last':
            last_hidden = hidden.squeeze(0)
        elif self.head_type == 'mean':
            # 모든 층의 값들의 평균을 구한다. 
            # lstm_out -> [batch_size, seq_len, hidden_size]
            last_hidden = torch.mean( lstm_out, dim=1 )    # [batch_size, hidden_size]
        elif self.head_type == 'max':
            last_hidden, _ = torch.max(lstm_out, dim = 1)

        dropout_hidden = self.dropout(last_hidden)

        # return self.fc(last_hidden)
        return self.fc(dropout_hidden)

In [13]:
model = LSTMCLF(len(vocab), emb_dim=64, hidden_size=128, num_classes=2, head_type='max')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [ ]:
#13.
epochs = 50

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct_train = 0 
    total_train = 0 

    for inputs, labels in tqdm(train_loader, desc = f"Epoch {epoch+1} / {epochs} "):
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pred = torch.argmax(output, dim=1)
        correct_train += (pred == labels).sum().item()
        total_train += labels.size(0)
    train_acc = (correct_train / total_train) * 100
    avg_train_loss  = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            output = model(inputs)
            loss = criterion(output, labels)

            val_loss += loss.item()
            pred = torch.argmax(output, dim = 1)
            correct_val += (pred == labels).sum().item()
            total_val += labels.size(0)

    val_acc = (correct_val / total_val) * 100
    avg_val_loss = val_loss / len(val_loader)

    if (epoch + 1) % 10 == 0:
        print(f"LSTM 에폭 결과 : Train Loss {round(avg_train_loss, 4)} Train Acc {train_acc} " )
        print(f"LSTM 에폭 결과 : Vali Loss {round(avg_val_loss, 4)} Vali Acc {val_acc}")